In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
import re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

In [ ]:
def preprocess_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+|#', '', text)  # Remove user mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    tokens = word_tokenize(text.lower())
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stopwords.words('english')]
    return " ".join(tokens)

In [ ]:
file_path = "unique_synthetic_joy_dataset.csv"  # Change this for each emotion
df = pd.read_csv(file_path)

df['preprocess_text'] = df['Sentence'].apply(preprocess_text)

emotion = 'joy'  # Replace with the target emotion
df['label'] = df['Label'].apply(lambda x: 1 if x == 'Yes' else 0)

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
tokens = tokenizer(
    list(df['preprocess_text']),
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Prepare dataset
labels = torch.tensor(df['label'].values)
dataset = TensorDataset(tokens['input_ids'], tokens['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
# Model setup
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = torch.nn.CrossEntropyLoss()

# Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(9):  # Adjust number of epochs
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}")

In [ ]:
def compute_confidence(logits):
    probabilities = torch.softmax(logits, dim=-1)
    return probabilities[:, 1]  # Confidence for the positive class (emotion present)

# Evaluation with confidence
model.eval()
preds, true_labels, confidences = [], [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        confidence_scores = compute_confidence(logits)

        # Apply threshold for predictions
        threshold = 0.7  # Adjust as needed
        batch_preds = (confidence_scores > threshold).int().cpu().numpy()

        preds.extend(batch_preds)
        true_labels.extend(labels.cpu().numpy())
        confidences.extend(confidence_scores.cpu().numpy())

accuracy = accuracy_score(true_labels, preds)
f1 = f1_score(true_labels, preds)
print(f"Validation Accuracy: {accuracy}, F1 Score: {f1}")

# Confidence output example
for i in range(5):  # Print first 5 predictions
    print(f"True Label: {true_labels[i]}, Prediction: {preds[i]}, Confidence: {confidences[i]:.5f}")

# Saving the model
model_save_path = "joy_classification_model.pth"  # Change for each emotion
tokenizer_save_path = "joy_tokenizer"  # Change for each emotion

torch.save(model.state_dict(), model_save_path)
tokenizer.save_pretrained(tokenizer_save_path)

print(f"Model saved to {model_save_path}")
print(f"Tokenizer saved to {tokenizer_save_path}")